# 8장 실습 ① — DNN으로 영상을 분류하면

**Keras 3 판**

2장에서 이렇게 말했습니다.

> *사과는 숫자 2개짜리 데이터이고, 손글씨 숫자는 숫자 784개짜리 데이터입니다.
> 그것 말고는 다를 것이 없습니다.*

**그 말이 어디까지 맞는지** 확인합니다.

## 8.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 8.1 도형 데이터

28×28 회색조 영상에 원·사각형·삼각형 중 하나가 그려져 있습니다.
**위치가 매번 다릅니다.** 그것이 이 장의 핵심입니다.

In [ ]:
# 도형 데이터 — 인터넷 없이 만든다. 그리고 **위치를 마음대로 흔들 수 있다.**
# MNIST는 숫자가 대체로 가운데 있어서 CNN이 왜 필요한지가 잘 안 드러난다.
x, y = data.shapes(n=6000, seed=42, shift=6)
s = data.split(x, y, val_ratio=0.15, test_ratio=0.15, seed=42)
print(s.summary())

fig = plot.image_grid(s.x_train, s.y_train, n=24, cols=8,
                      class_names=list(data.SHAPE_CLASSES))
plt.show()

## 8.2 학습 함수 — 여기만 판마다 다릅니다

`kind` 로 DNN / CNN / 선형 모델을 만듭니다.
**PyTorch 판에서 `permute` 한 줄이 더 있는 것**에 주목하십시오 — 채널 순서가
다르기 때문입니다.

In [ ]:
import keras
from keras import layers

dlbook.set_seed(42)

def train(kind, split=None, units=(256, 128), conv_act="relu", pool="max",
          head_act="relu", n_classes=3, epochs=15, bs=64, lr=0.001, seed=42):
    """모델을 만들어 학습시키고 (시험 정확도, 파라미터 수)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    sp = split if split is not None else s
    dlbook.set_seed(seed)
    shape = sp.x_train.shape[1:]

    if kind == "dnn":
        ls = [layers.Input(shape=shape), layers.Flatten()] + \
             [layers.Dense(u, activation="relu") for u in units]
    elif kind == "linear":
        ls = [layers.Input(shape=shape), layers.Flatten()]
    else:                                        # cnn
        Pool = layers.MaxPooling2D if pool == "max" else layers.AveragePooling2D
        ls = [layers.Input(shape=shape)]
        for f in (16, 32):
            ls.append(layers.Conv2D(f, 3, activation=conv_act, padding="same"))
            ls.append(Pool(2))
        ls += [layers.Flatten(), layers.Dense(64, activation=head_act)]
    ls.append(layers.Dense(n_classes, activation="softmax"))

    model = keras.Sequential(ls)
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(sp.x_train, sp.y_train, validation_data=(sp.x_val, sp.y_val),
              epochs=dlbook.smoke.epochs(epochs), batch_size=bs, verbose=0)
    pred = model.predict(sp.x_test, verbose=0).argmax(1)
    return metrics.accuracy(sp.y_test, pred), model.count_params()

## 8.3 DNN과 CNN을 나란히

같은 데이터, 같은 epoch. **구조만 다릅니다.**

In [ ]:
print(f"{'모델':<24}{'파라미터':>12}{'시험 정확도':>12}")
for name, kind, kw in (("DNN (256-128)", "dnn", {}),
                       ("DNN (512-256-128)", "dnn", {"units": (512, 256, 128)}),
                       ("CNN (16-32)", "cnn", {})):
    acc, n_params = train(kind, **kw)
    print(f"{name:<24}{n_params:>12,}{acc:>12.3f}")
    dlbook.record(f"ch08_{name.split()[0].lower()}_{len(kw)}_acc", acc)

print()
print("→ CNN이 **파라미터는 절반인데** 성능은 훨씬 낫습니다.")
print("→ 모델을 키워서 될 문제가 아니었습니다. 구조 문제였습니다.")

## 8.4 위치를 얼마나 흔드느냐

DNN이 왜 무너지는지가 여기서 드러납니다.

In [ ]:
# 도형의 위치를 얼마나 흔드느냐만 바꾼다.
shifts = [1, 6] if dlbook.smoke.is_smoke() else [1, 3, 6]
print(f"{'위치 흔들림':<14}{'DNN':>10}{'CNN':>10}{'차이':>10}")
for sh in shifts:
    xs, ys = data.shapes(6000, seed=42, shift=sh)
    ss = data.split(xs, ys, val_ratio=0.15, test_ratio=0.15, seed=42)
    a, _ = train("dnn", split=ss)
    b, _ = train("cnn", split=ss)
    print(f"{'±' + str(sh) + ' 픽셀':<14}{a:>10.3f}{b:>10.3f}{b - a:>10.3f}")
    dlbook.record(f"ch08_shift{sh}_dnn_acc", a)
    dlbook.record(f"ch08_shift{sh}_cnn_acc", b)

print()
print("→ DNN은 위치가 고정이면 잘합니다. 흔들리면 무너집니다.")
print("→ Flatten()이 '어느 픽셀이 어느 픽셀 옆이었는지'를 버리기 때문입니다.")

## 정리

- 2장의 방식(`Flatten` → `Dense`)도 **위치가 고정이면 잘 됩니다.**
  **위치가 흔들리면 무너집니다.**
- `Flatten()` 이 **이웃 관계를 버리기** 때문입니다. 모든 위치를 따로 배워야 합니다.
- **합성곱은 같은 커널로 온 화면을 훑습니다.** 한 번 배운 것이 모든 위치에서
  통합니다. 그래서 파라미터가 훨씬 적고 성능은 훨씬 낫습니다.
- **모델을 키워서 될 문제가 아니라 구조 문제였습니다.**

### 연습

1. `shift` 를 0, 2, 4, 8로 바꿔 가며 두 곡선을 한 그림에 그리십시오.
2. CNN의 첫 Conv2D 층이 학습한 **커널 16장을 그림으로** 그려 보십시오.
3. CNN에서 `MaxPooling2D` 를 빼면 파라미터와 성능이 어떻게 됩니까.